In [0]:
from pyspark.sql import functions as F

# Create dim_store dimension table
store_df = spark.table("automobile_catalog.002_silver.store")

dim_store = store_df.select(
    "store_id",
    "store_name",
    "city",
    "state",
    "manager_id",
    "manager_name",
    "opened_year",
    "store_type"
).distinct()

# Write to gold layer
dim_store.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.dim_store")

print(f"dim_store created with {dim_store.count()} records")
display(dim_store)

In [0]:
from pyspark.sql import functions as F

# Create dim_technician dimension table
orders_df = spark.table("automobile_catalog.002_silver.order")
store_df = spark.table("automobile_catalog.002_silver.store")

dim_technician = (
    orders_df
    .join(store_df, "store_id", "inner")
    .select(
        "technician_id",
        "technician_name",
        "store_id",
        "store_name"
    )
    .distinct()
)

# Write to gold layer
dim_technician.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.dim_technician")

print(f"dim_technician created with {dim_technician.count()} records")
display(dim_technician)

In [0]:
from pyspark.sql import functions as F

# Create dim_estimator dimension table
estimate_df = spark.table("automobile_catalog.002_silver.estimate")

dim_estimator = (
    estimate_df
    .select(
        "estimator_id",
        "estimator_name"
    )
    .distinct()
)

# Write to gold layer
dim_estimator.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.dim_estimator")

print(f"dim_estimator created with {dim_estimator.count()} records")
display(dim_estimator)

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta, date

# Get date range from orders and invoices
orders_df = spark.table("automobile_catalog.002_silver.order")
invoices_df = spark.table("automobile_catalog.002_silver.invoice")

# Find min and max dates
min_date = orders_df.select(F.min(F.col("vehicle_in_datetime"))).first()[0]
max_date = invoices_df.select(F.max(F.col("invoice_date"))).first()[0]

if min_date and max_date:
    # Convert to date objects (handle both datetime and date types)
    if isinstance(min_date, datetime):
        current_date = min_date.date()
    else:
        current_date = min_date
    
    if isinstance(max_date, datetime):
        end_date = max_date.date()
    else:
        end_date = max_date
    
    # Create date dimension
    date_list = []
    
    while current_date <= end_date:
        date_list.append({
            "date_key": int(current_date.strftime("%Y%m%d")),
            "date": current_date,
            "year": current_date.year,
            "quarter": (current_date.month - 1) // 3 + 1,
            "month": current_date.month,
            "month_name": current_date.strftime("%B"),
            "day": current_date.day,
            "day_of_week": current_date.isoweekday(),
            "day_name": current_date.strftime("%A"),
            "week_of_year": current_date.isocalendar()[1],
            "is_weekend": current_date.isoweekday() >= 6
        })
        current_date += timedelta(days=1)
    
    dim_date = spark.createDataFrame(date_list)
    
    # Write to gold layer
    dim_date.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.dim_date")
    
    print(f"dim_date created with {dim_date.count()} records")
    display(dim_date.orderBy("date").limit(10))
else:
    print("No date data available")

In [0]:
from pyspark.sql import functions as F

# Read silver tables
orders_df = spark.table("automobile_catalog.002_silver.order")
store_df = spark.table("automobile_catalog.002_silver.store")

# Create fact_orders with calculated metrics
fact_orders = (
    orders_df
    .join(store_df, "store_id", "inner")
    .withColumn("vehicle_in_date_key", F.date_format("vehicle_in_datetime", "yyyyMMdd").cast("int"))
    .withColumn("vehicle_out_date_key", F.date_format("vehicle_out_datetime", "yyyyMMdd").cast("int"))
    .withColumn("promised_delivery_date_key", F.date_format("promised_delivery_datetime", "yyyyMMdd").cast("int"))
    .withColumn("actual_delivery_date_key", F.date_format("actual_delivery_datetime", "yyyyMMdd").cast("int"))
    .withColumn(
        "days_in_shop",
        F.when(
            F.col("vehicle_in_datetime").isNotNull() & F.col("vehicle_out_datetime").isNotNull(),
            F.datediff(F.col("vehicle_out_datetime"), F.col("vehicle_in_datetime"))
        ).otherwise(None)
    )
    .withColumn(
        "days_to_work_start",
        F.when(
            F.col("vehicle_in_datetime").isNotNull() & F.col("actual_work_start_datetime").isNotNull(),
            F.datediff(F.col("actual_work_start_datetime"), F.col("vehicle_in_datetime"))
        ).otherwise(None)
    )
    .withColumn(
        "work_duration_days",
        F.when(
            F.col("actual_work_start_datetime").isNotNull() & F.col("actual_completion_datetime").isNotNull(),
            F.datediff(F.col("actual_completion_datetime"), F.col("actual_work_start_datetime"))
        ).otherwise(None)
    )
    .withColumn(
        "delivery_variance_days",
        F.when(
            F.col("promised_delivery_datetime").isNotNull() & F.col("actual_delivery_datetime").isNotNull(),
            F.datediff(F.col("actual_delivery_datetime"), F.col("promised_delivery_datetime"))
        ).otherwise(None)
    )
    .withColumn(
        "is_on_time",
        F.when(
            F.col("actual_delivery_datetime") <= F.col("promised_delivery_datetime"),
            True
        ).otherwise(False)
    )
    .select(
        "order_id",
        "vehicle_in_date_key",
        "vehicle_out_date_key",
        "promised_delivery_date_key",
        "actual_delivery_date_key",
        "store_id",
        "manager_id",
        "technician_id",
        "service_type",
        "order_status",
        "customer_name",
        "customer_phone",
        "vehicle_no",
        "vehicle_make",
        "vehicle_model",
        "days_in_shop",
        "days_to_work_start",
        "work_duration_days",
        "delivery_variance_days",
        "is_on_time",
        "vehicle_in_datetime",
        "vehicle_out_datetime",
        "promised_delivery_datetime",
        "actual_delivery_datetime"
    )
)

# Write to gold layer
fact_orders.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.fact_orders")

print(f"fact_orders created with {fact_orders.count()} records")
display(fact_orders.limit(10))

In [0]:
from pyspark.sql import functions as F

# Read silver tables
invoices_df = spark.table("automobile_catalog.002_silver.invoice")
orders_df = spark.table("automobile_catalog.002_silver.order")
store_df = spark.table("automobile_catalog.002_silver.store")

# Create fact_invoices
fact_invoices = (
    invoices_df
    .join(orders_df, "order_id", "inner")
    .join(store_df, "store_id", "inner")
    .withColumn("invoice_date_key", F.date_format("invoice_date", "yyyyMMdd").cast("int"))
    .withColumn("invoice_month", F.date_format("invoice_date", "yyyy-MM"))
    .withColumn("invoice_year", F.year("invoice_date"))
    .withColumn("invoice_quarter", F.quarter("invoice_date"))
    .select(
        "invoice_id",
        "order_id",
        "invoice_date_key",
        "invoice_date",
        "invoice_month",
        "invoice_year",
        "invoice_quarter",
        "store_id",
        "manager_id",
        "technician_id",
        "customer_name",
        "customer_phone",
        "service_type",
        "order_status",
        "invoice_amount",
        "payment_mode"
    )
)

# Write to gold layer
fact_invoices.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.fact_invoices")

print(f"fact_invoices created with {fact_invoices.count()} records")
display(fact_invoices.limit(10))

In [0]:
from pyspark.sql import functions as F

# Read silver tables
estimate_df = spark.table("automobile_catalog.002_silver.estimate")
orders_df = spark.table("automobile_catalog.002_silver.order")
store_df = spark.table("automobile_catalog.002_silver.store")
invoices_df = spark.table("automobile_catalog.002_silver.invoice")

# Create fact_estimates with accuracy metrics
fact_estimates = (
    estimate_df
    .join(orders_df, "order_id", "inner")
    .join(store_df, "store_id", "inner")
    .join(invoices_df.select("order_id", F.col("invoice_amount").alias("actual_amount")), "order_id", "left")
    .withColumn("estimate_date_key", F.date_format("created_at", "yyyyMMdd").cast("int"))
    .withColumn("estimate_date", F.to_date("created_at"))
    .withColumn(
        "estimate_variance",
        F.when(
            F.col("actual_amount").isNotNull(),
            F.abs(F.col("estimate_amount") - F.col("actual_amount"))
        ).otherwise(None)
    )
    .withColumn(
        "variance_pct",
        F.when(
            F.col("actual_amount").isNotNull() & (F.col("actual_amount") > 0),
            F.round((F.abs(F.col("estimate_amount") - F.col("actual_amount")) / F.col("actual_amount")) * 100, 2)
        ).otherwise(None)
    )
    .withColumn(
        "is_initial_estimate",
        F.when(F.col("version_no") == 1, True).otherwise(False)
    )
    .select(
        "estimate_id",
        "order_id",
        "estimate_date_key",
        "estimate_date",
        "store_id",
        "manager_id",
        "technician_id",
        "estimator_id",
        "customer_name",
        "customer_phone",
        "service_type",
        "version_no",
        "is_initial_estimate",
        "estimate_amount",
        "actual_amount",
        "estimate_variance",
        "variance_pct"
    )
)

# Write to gold layer
fact_estimates.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.fact_estimates")

print(f"fact_estimates created with {fact_estimates.count()} records")
display(fact_estimates.limit(10))

In [0]:
from pyspark.sql import functions as F

# Read silver tables
survey_df = spark.table("automobile_catalog.002_silver.customer_survey")
orders_df = spark.table("automobile_catalog.002_silver.order")
store_df = spark.table("automobile_catalog.002_silver.store")

# Create fact_survey_responses
fact_survey_responses = (
    survey_df
    .join(orders_df, "order_id", "inner")
    .join(store_df, "store_id", "inner")
    .withColumn("survey_sent_date_key", F.date_format("survey_sent_date", "yyyyMMdd").cast("int"))
    .withColumn("survey_response_date_key", F.date_format("survey_response_date", "yyyyMMdd").cast("int"))
    .withColumn(
        "response_time_days",
        F.when(
            F.col("survey_response_date").isNotNull(),
            F.datediff(F.col("survey_response_date"), F.col("survey_sent_date"))
        ).otherwise(None)
    )
    .withColumn(
        "avg_rating",
        F.when(
            F.col("responded_flag") == True,
            F.round(
                (F.coalesce(F.col("delivered_on_time_rating"), F.lit(0)) +
                 F.coalesce(F.col("work_quality_rating"), F.lit(0)) +
                 F.coalesce(F.col("cleanliness_rating"), F.lit(0)) +
                 F.coalesce(F.col("communication_rating"), F.lit(0))) / 4,
                2
            )
        ).otherwise(None)
    )
    .select(
        "survey_id",
        "order_id",
        "survey_sent_date_key",
        "survey_response_date_key",
        "survey_sent_date",
        "survey_response_date",
        "store_id",
        "manager_id",
        "technician_id",
        "customer_name",
        "customer_phone",
        "service_type",
        "responded_flag",
        "response_time_days",
        "delivered_on_time_rating",
        "work_quality_rating",
        "cleanliness_rating",
        "communication_rating",
        "overall_satisfaction_rating",
        "avg_rating"
    )
)

# Write to gold layer
fact_survey_responses.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.fact_survey_responses")

print(f"fact_survey_responses created with {fact_survey_responses.count()} records")
display(fact_survey_responses.limit(10))

In [0]:
from pyspark.sql import functions as F

# Read silver tables
budget_df = spark.table("automobile_catalog.002_silver.ns_budget")

# Create fact_budget
fact_budget = (
    budget_df
    .withColumn("budget_date", F.col("month"))  # month is already a date
    .withColumn("budget_date_key", F.date_format("month", "yyyyMMdd").cast("int"))
    .withColumn("budget_month", F.date_format("month", "yyyy-MM"))
    .withColumn("budget_year", F.year("month"))
    .withColumn("budget_quarter", F.quarter("month"))
    .select(
        "ns_store_id",
        F.col("ns_store_id").alias("store_id"),  # Align with store_id naming convention
        "budget_date_key",
        "budget_date",
        "budget_month",
        "budget_year",
        "budget_quarter",
        "budget_amount"
    )
)

# Write to gold layer
fact_budget.write.mode("overwrite").saveAsTable("automobile_catalog.003_gold.fact_budget")

print(f"fact_budget created with {fact_budget.count()} records")
display(fact_budget.limit(10))